In [1]:
!pip install -q ultralytics onnx onnxsim

import json, os, shutil, random, yaml
from pathlib import Path
from collections import defaultdict, Counter
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you h

In [3]:
BASE = "/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease"

JSON_PATH = f"{BASE}/train_quadrant_enumeration_disease.json"
IMG_SRC = f"{BASE}/xrays"

WORK = "/kaggle/working"
FDI_OUT = f"{WORK}/dentex_fdi_yolo"
DISEASE_OUT = f"{WORK}/dentex_disease_yolo"

print(os.path.exists(JSON_PATH), JSON_PATH)
print(os.path.exists(IMG_SRC), IMG_SRC)

True /kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json
True /kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023/training_data/training_data/quadrant-enumeration-disease/xrays


In [4]:
with open(JSON_PATH) as f:
    data = json.load(f)

print(data.keys())
print("Images:", len(data["images"]))
print("Annotations:", len(data["annotations"]))
print("Esempio annotation:")
print(data["annotations"][0])

dict_keys(['images', 'annotations', 'categories_1', 'categories_2', 'categories_3'])
Images: 705
Annotations: 3529
Esempio annotation:
{'iscrowd': 0, 'image_id': 1, 'bbox': [542.0, 698.0, 220.0, 271.0], 'segmentation': [[621, 703, 573, 744, 542, 885, 580, 945, 650, 969, 711, 883, 762, 807, 748, 741, 649, 698]], 'id': 1, 'area': 39683, 'category_id_1': 3, 'category_id_2': 7, 'category_id_3': 0}


In [5]:
images = {img["id"]: img for img in data["images"]}

anns_by_image = defaultdict(list)
for ann in data["annotations"]:
    anns_by_image[ann["image_id"]].append(ann)

random.seed(42)
all_ids = list(images.keys())
random.shuffle(all_ids)

cut = int(len(all_ids) * 0.9)
train_ids = set(all_ids[:cut])
val_ids = set(all_ids[cut:])

print("Train:", len(train_ids))
print("Val:", len(val_ids))

Train: 634
Val: 71


In [6]:
# FDI permanenti: 11-18, 21-28, 31-38, 41-48
fdi_list = []
for q in [1, 2, 3, 4]:
    for t in [1, 2, 3, 4, 5, 6, 7, 8]:
        fdi_list.append(f"{q}{t}")

fdi_to_class = {fdi: idx for idx, fdi in enumerate(fdi_list)}
class_to_fdi = {idx: fdi for fdi, idx in fdi_to_class.items()}

print(class_to_fdi)

{0: '11', 1: '12', 2: '13', 3: '14', 4: '15', 5: '16', 6: '17', 7: '18', 8: '21', 9: '22', 10: '23', 11: '24', 12: '25', 13: '26', 14: '27', 15: '28', 16: '31', 17: '32', 18: '33', 19: '34', 20: '35', 21: '36', 22: '37', 23: '38', 24: '41', 25: '42', 26: '43', 27: '44', 28: '45', 29: '46', 30: '47', 31: '48'}


In [7]:
shutil.rmtree(FDI_OUT, ignore_errors=True)

for split in ["train", "val"]:
    Path(f"{FDI_OUT}/images/{split}").mkdir(parents=True, exist_ok=True)
    Path(f"{FDI_OUT}/labels/{split}").mkdir(parents=True, exist_ok=True)

for img_id, info in images.items():
    split = "train" if img_id in train_ids else "val"

    filename = info["file_name"]
    img_w = info["width"]
    img_h = info["height"]

    shutil.copy2(f"{IMG_SRC}/{filename}", f"{FDI_OUT}/images/{split}/{filename}")

    label_path = f"{FDI_OUT}/labels/{split}/{Path(filename).stem}.txt"

    with open(label_path, "w") as lf:
        for ann in anns_by_image[img_id]:
            quadrant = ann["category_id_1"] + 1
            tooth_number = ann["category_id_2"] + 1
            fdi = f"{quadrant}{tooth_number}"

            if fdi not in fdi_to_class:
                continue

            cls = fdi_to_class[fdi]

            x, y, w, h = ann["bbox"]

            xc = (x + w / 2) / img_w
            yc = (y + h / 2) / img_h
            wn = w / img_w
            hn = h / img_h

            lf.write(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

print("Dataset FDI creato:", FDI_OUT)

Dataset FDI creato: /kaggle/working/dentex_fdi_yolo


In [7]:
fdi_yaml = {
    "path": FDI_OUT,
    "train": "images/train",
    "val": "images/val",
    "names": class_to_fdi
}

with open(f"{WORK}/fdi.yaml", "w") as f:
    yaml.dump(fdi_yaml, f, sort_keys=False)

print(open(f"{WORK}/fdi.yaml").read())

path: /kaggle/working/dentex_fdi_yolo
train: images/train
val: images/val
names:
  0: '11'
  1: '12'
  2: '13'
  3: '14'
  4: '15'
  5: '16'
  6: '17'
  7: '18'
  8: '21'
  9: '22'
  10: '23'
  11: '24'
  12: '25'
  13: '26'
  14: '27'
  15: '28'
  16: '31'
  17: '32'
  18: '33'
  19: '34'
  20: '35'
  21: '36'
  22: '37'
  23: '38'
  24: '41'
  25: '42'
  26: '43'
  27: '44'
  28: '45'
  29: '46'
  30: '47'
  31: '48'



In [8]:
# Mapping DentalCare:
# 0 -> Impacted
# 1 -> Caries
# 2 -> Periapical_Lesion
# 3 -> Deep_Caries

disease_names = {
    0: "Impacted",
    1: "Caries",
    2: "Periapical_Lesion",
    3: "Deep_Caries"
}

print("Disease category counts:")
print(Counter([ann["category_id_3"] for ann in data["annotations"]]))

shutil.rmtree(DISEASE_OUT, ignore_errors=True)

for split in ["train", "val"]:
    Path(f"{DISEASE_OUT}/images/{split}").mkdir(parents=True, exist_ok=True)
    Path(f"{DISEASE_OUT}/labels/{split}").mkdir(parents=True, exist_ok=True)

for img_id, info in images.items():
    split = "train" if img_id in train_ids else "val"

    filename = info["file_name"]
    img_w = info["width"]
    img_h = info["height"]

    shutil.copy2(f"{IMG_SRC}/{filename}", f"{DISEASE_OUT}/images/{split}/{filename}")

    label_path = f"{DISEASE_OUT}/labels/{split}/{Path(filename).stem}.txt"

    with open(label_path, "w") as lf:
        for ann in anns_by_image[img_id]:
            cls = ann["category_id_3"]

            x, y, w, h = ann["bbox"]

            xc = (x + w / 2) / img_w
            yc = (y + h / 2) / img_h
            wn = w / img_w
            hn = h / img_h

            lf.write(f"{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

print("Dataset disease creato:", DISEASE_OUT)

Disease category counts:
Counter({1: 2189, 0: 604, 3: 578, 2: 158})
Dataset disease creato: /kaggle/working/dentex_disease_yolo


In [9]:
disease_yaml = {
    "path": DISEASE_OUT,
    "train": "images/train",
    "val": "images/val",
    "names": disease_names
}

with open(f"{WORK}/disease.yaml", "w") as f:
    yaml.dump(disease_yaml, f, sort_keys=False)

print(open(f"{WORK}/disease.yaml").read())

path: /kaggle/working/dentex_disease_yolo
train: images/train
val: images/val
names:
  0: Impacted
  1: Caries
  2: Periapical_Lesion
  3: Deep_Caries



In [10]:
print("FDI train images:", len(list(Path(f"{FDI_OUT}/images/train").glob("*"))))
print("FDI val images:", len(list(Path(f"{FDI_OUT}/images/val").glob("*"))))
print("Disease train images:", len(list(Path(f"{DISEASE_OUT}/images/train").glob("*"))))
print("Disease val images:", len(list(Path(f"{DISEASE_OUT}/images/val").glob("*"))))

print("Esempio label FDI:")
print(list(Path(f"{FDI_OUT}/labels/train").glob("*.txt"))[0].read_text()[:500])

print("Esempio label disease:")
print(list(Path(f"{DISEASE_OUT}/labels/train").glob("*.txt"))[0].read_text()[:500])

FDI train images: 634
FDI val images: 71
Disease train images: 634
Disease val images: 71
Esempio label FDI:
3 0.405999 0.400180 0.033788 0.323294
4 0.381136 0.398107 0.029963 0.313623
5 0.354998 0.405015 0.041438 0.261122
6 0.316110 0.398798 0.042713 0.218292
11 0.587690 0.418831 0.035063 0.291517
22 0.694457 0.608352 0.080358 0.211868
28 0.387616 0.680768 0.042268 0.237332
30 0.313407 0.626119 0.083095 0.202981
31 0.277863 0.572511 0.085977 0.168631
22 0.688464 0.601824 0.067672 0.199088

Esempio label disease:
3 0.405999 0.400180 0.033788 0.323294
3 0.381136 0.398107 0.029963 0.313623
1 0.354998 0.405015 0.041438 0.261122
1 0.316110 0.398798 0.042713 0.218292
3 0.587690 0.418831 0.035063 0.291517
2 0.694457 0.608352 0.080358 0.211868
1 0.387616 0.680768 0.042268 0.237332
3 0.313407 0.626119 0.083095 0.202981
1 0.277863 0.572511 0.085977 0.168631
3 0.688464 0.601824 0.067672 0.199088



In [15]:
model_fdi = YOLO("yolov8l.pt")

model_fdi.train(
    data=f"{WORK}/fdi.yaml",
    epochs=100,
    imgsz=1024,
    batch=4,
    project=f"{WORK}/runs",
    name="dentex_fdi_v1",
    patience=20,
    workers=2
)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/fdi.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dentex_fdi_v1-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f5fb50dbbc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004

In [22]:
from ultralytics import YOLO

disease_best = "/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt"

model_disease = YOLO(disease_best)

disease_metrics = model_disease.val(
    data="/kaggle/working/disease.yaml",
    imgsz=1024,
    batch=4
)

print(disease_metrics)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3749.6±1676.1 MB/s, size: 4718.1 KB)
val: Scanning /kaggle/working/dentex_disease_yolo/labels/val.cache... 71 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 71/71 17.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 5.2it/s 3.5s0.2s
                   all         71        363      0.728      0.555      0.605      0.433
              Impacted         25         64      0.832      0.931      0.923       0.62
                Caries         61        246      0.512      0.418      0.486      0.352
     Periapical_Lesion         11         14      0.767      0.357      0.403      0.329
           Deep_Caries         25         39      0.801      0.513      0.609       0.43
Speed: 2.2ms preprocess, 1

In [ ]:
model_fdi.predict(
    source=f"{FDI_OUT}/images/val",
    imgsz=1024,
    conf=0.25,
    save=True,
    project=f"{WORK}/predictions",
    name="fdi_predictions"
)

In [16]:
model_disease = YOLO("yolov8s.pt")

model_disease.train(
    data=f"{WORK}/disease.yaml",
    epochs=100,
    imgsz=1024,
    batch=4,
    project=f"{WORK}/runs",
    name="dentex_disease_v1",
    patience=25,
    workers=2
)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/disease.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dentex_disease_v1-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f5f4fc1f4d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [25]:
from pathlib import Path
import os, time

runs_dir = Path("/kaggle/working/runs")

pt_files = list(runs_dir.glob("**/weights/*.pt"))

print("MODELLI .pt TROVATI:")
for p in sorted(pt_files, key=lambda x: x.stat().st_mtime):
    size_mb = p.stat().st_size / 1024 / 1024
    modified = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(p.stat().st_mtime))
    print(f"{p} | {size_mb:.2f} MB | modified: {modified}")

MODELLI .pt TROVATI:
/kaggle/working/runs/dentex_disease_v1/weights/last.pt | 85.40 MB | modified: 2026-06-30 07:54:38
/kaggle/working/runs/dentex_disease_v1/weights/best.pt | 0.00 MB | modified: 2026-06-30 07:54:38
/kaggle/working/runs/dentex_fdi_v1/weights/best.pt | 85.49 MB | modified: 2026-06-30 08:12:00
/kaggle/working/runs/dentex_fdi_v1/weights/last.pt | 85.49 MB | modified: 2026-06-30 08:15:22
/kaggle/working/runs/dentex_fdi_v1-2/weights/last.pt | 83.66 MB | modified: 2026-06-30 09:43:18
/kaggle/working/runs/dentex_fdi_v1-2/weights/best.pt | 83.66 MB | modified: 2026-06-30 09:43:19
/kaggle/working/runs/dentex_disease_v1-3/weights/last.pt | 21.51 MB | modified: 2026-06-30 11:06:29
/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt | 21.51 MB | modified: 2026-06-30 11:06:30


In [26]:
from ultralytics import YOLO

FDI_BEST = "/kaggle/working/runs/dentex_fdi_v1-2/weights/best.pt"
DISEASE_BEST = "/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt"

fdi_model = YOLO(FDI_BEST)
disease_model = YOLO(DISEASE_BEST)

print("VALIDAZIONE FDI")
fdi_metrics = fdi_model.val(
    data="/kaggle/working/fdi.yaml",
    imgsz=1024,
    batch=4,
    project="/kaggle/working/validation",
    name="fdi_val"
)

print("VALIDAZIONE DISEASE")
disease_metrics = disease_model.val(
    data="/kaggle/working/disease.yaml",
    imgsz=1024,
    batch=4,
    project="/kaggle/working/validation",
    name="disease_val"
)

VALIDAZIONE FDI
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 113 layers, 43,631,280 parameters, 0 gradients, 165.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4643.6±657.9 MB/s, size: 4220.6 KB)
val: Scanning /kaggle/working/dentex_fdi_yolo/labels/val.cache... 71 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 71/71 33.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s0.3s
                   all         71        363      0.575      0.473      0.472      0.337
                    11          6          6      0.261      0.333      0.394       0.31
                    12          4          4      0.674      0.523      0.754      0.525
                    13          2          2          1          0      0.055      0.044
                    14         10         10      0.652        0.4      0.405      0.316
             

In [27]:
from ultralytics import YOLO
from pathlib import Path

FDI_BEST = "/kaggle/working/runs/dentex_fdi_v1-2/weights/best.pt"
DISEASE_BEST = "/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt"

FDI_VAL_IMAGES = "/kaggle/working/dentex_fdi_yolo/images/val"
DISEASE_VAL_IMAGES = "/kaggle/working/dentex_disease_yolo/images/val"

fdi_model = YOLO(FDI_BEST)
disease_model = YOLO(DISEASE_BEST)

# Prendiamo 10 immagini di validation
sample_images = sorted(list(Path(FDI_VAL_IMAGES).glob("*")))[:10]

print("Immagini usate:")
for img in sample_images:
    print(img)

fdi_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=1024,
    conf=0.25,
    save=True,
    project="/kaggle/working/predictions",
    name="fdi_sample"
)

disease_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=1024,
    conf=0.25,
    save=True,
    project="/kaggle/working/predictions",
    name="disease_sample"
)

Immagini usate:
/kaggle/working/dentex_fdi_yolo/images/val/train_127.png
/kaggle/working/dentex_fdi_yolo/images/val/train_148.png
/kaggle/working/dentex_fdi_yolo/images/val/train_171.png
/kaggle/working/dentex_fdi_yolo/images/val/train_172.png
/kaggle/working/dentex_fdi_yolo/images/val/train_177.png
/kaggle/working/dentex_fdi_yolo/images/val/train_182.png
/kaggle/working/dentex_fdi_yolo/images/val/train_201.png
/kaggle/working/dentex_fdi_yolo/images/val/train_220.png
/kaggle/working/dentex_fdi_yolo/images/val/train_224.png
/kaggle/working/dentex_fdi_yolo/images/val/train_235.png

0: 1024x1024 1 25, 1 26, 1 48, 108.8ms
1: 1024x1024 1 18, 1 24, 1 25, 1 26, 1 28, 1 35, 1 36, 1 38, 1 45, 1 46, 1 47, 1 48, 108.8ms
2: 1024x1024 1 16, 1 17, 1 18, 1 26, 1 27, 1 34, 1 35, 1 46, 1 47, 108.8ms
3: 1024x1024 1 26, 1 38, 1 46, 2 47s, 108.8ms
4: 1024x1024 1 34, 1 36, 108.8ms
5: 1024x1024 1 15, 1 18, 1 27, 1 37, 1 46, 1 47, 108.8ms
6: 1024x1024 1 16, 1 17, 1 28, 1 37, 1 38, 1 46, 1 48, 108.8ms
7: 1024

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Impacted', 1: 'Caries', 2: 'Periapical_Lesion', 3: 'Deep_Caries'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        ...,
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
 

In [28]:
print("FDI predictions:")
!ls -lah /kaggle/working/predictions/fdi_sample

print("Disease predictions:")
!ls -lah /kaggle/working/predictions/disease_sample

FDI predictions:
total 9.6M
drwxr-xr-x 2 root root 4.0K Jun 30 11:31 .
drwxr-xr-x 4 root root 4.0K Jun 30 11:31 ..
-rw-r--r-- 1 root root 837K Jun 30 11:31 image0.jpg
-rw-r--r-- 1 root root 1.1M Jun 30 11:31 image1.jpg
-rw-r--r-- 1 root root 1.2M Jun 30 11:31 image2.jpg
-rw-r--r-- 1 root root 1.1M Jun 30 11:31 image3.jpg
-rw-r--r-- 1 root root 799K Jun 30 11:31 image4.jpg
-rw-r--r-- 1 root root 1.1M Jun 30 11:31 image5.jpg
-rw-r--r-- 1 root root 993K Jun 30 11:31 image6.jpg
-rw-r--r-- 1 root root 885K Jun 30 11:31 image7.jpg
-rw-r--r-- 1 root root 996K Jun 30 11:31 image8.jpg
-rw-r--r-- 1 root root 967K Jun 30 11:31 image9.jpg
Disease predictions:
total 9.8M
drwxr-xr-x 2 root root  4.0K Jun 30 11:31 .
drwxr-xr-x 4 root root  4.0K Jun 30 11:31 ..
-rw-r--r-- 1 root root  834K Jun 30 11:31 image0.jpg
-rw-r--r-- 1 root root  1.2M Jun 30 11:31 image1.jpg
-rw-r--r-- 1 root root  1.2M Jun 30 11:31 image2.jpg
-rw-r--r-- 1 root root  1.1M Jun 30 11:31 image3.jpg
-rw-r--r-- 1 root root  795K Jun

In [29]:
from ultralytics import YOLO
import shutil
from pathlib import Path

FDI_BEST = "/kaggle/working/runs/dentex_fdi_v1-2/weights/best.pt"
DISEASE_BEST = "/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt"

fdi_model = YOLO(FDI_BEST)
disease_model = YOLO(DISEASE_BEST)

print("Export FDI ONNX...")
fdi_model.export(
    format="onnx",
    imgsz=1024,
    simplify=True
)

print("Export disease ONNX...")
disease_model.export(
    format="onnx",
    imgsz=1024,
    simplify=True
)

Export FDI ONNX...
Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 113 layers, 43,631,280 parameters, 0 gradients, 165.0 GFLOPs

PyTorch: starting from '/kaggle/working/runs/dentex_fdi_v1-2/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 36, 21504) (83.7 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 462ms
Prepared 2 packages in 395ms
Installed 2 packages in 41ms
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 1.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ON

'/kaggle/working/runs/dentex_disease_v1-3/weights/best.onnx'

In [30]:
from pathlib import Path

onnx_files = list(Path("/kaggle/working/runs").glob("**/*.onnx"))

for p in onnx_files:
    print(p, "|", round(p.stat().st_size / 1024 / 1024, 2), "MB")

/kaggle/working/runs/dentex_disease_v1-3/weights/best.onnx | 42.93 MB
/kaggle/working/runs/dentex_fdi_v1-2/weights/best.onnx | 166.96 MB


In [31]:
import shutil
from pathlib import Path

EXPORT = Path("/kaggle/working/dentalcare_models")
EXPORT.mkdir(exist_ok=True)

shutil.copy2(
    "/kaggle/working/runs/dentex_fdi_v1-2/weights/best.onnx",
    EXPORT / "dentex_fdi_v1.onnx"
)

shutil.copy2(
    "/kaggle/working/runs/dentex_disease_v1-3/weights/best.onnx",
    EXPORT / "dentex_disease_v1.onnx"
)

print("Contenuto export:")

for f in EXPORT.iterdir():
    print(f.name, round(f.stat().st_size/1024/1024,2),"MB")

Contenuto export:
dentex_fdi_v1.onnx 166.96 MB
dentex_disease_v1.onnx 42.93 MB


In [32]:
import onnx

for model in [
    "/kaggle/working/dentalcare_models/dentex_fdi_v1.onnx",
    "/kaggle/working/dentalcare_models/dentex_disease_v1.onnx"
]:
    m = onnx.load(model)
    onnx.checker.check_model(m)

    print(model)
    print("OK")

/kaggle/working/dentalcare_models/dentex_fdi_v1.onnx
OK
/kaggle/working/dentalcare_models/dentex_disease_v1.onnx
OK


In [33]:
import onnx

for model in [
    "/kaggle/working/dentalcare_models/dentex_fdi_v1.onnx",
    "/kaggle/working/dentalcare_models/dentex_disease_v1.onnx"
]:
    m = onnx.load(model)
    onnx.checker.check_model(m)
    print(model, "OK")

/kaggle/working/dentalcare_models/dentex_fdi_v1.onnx OK
/kaggle/working/dentalcare_models/dentex_disease_v1.onnx OK


In [34]:
import onnxruntime as ort

for model in [
    "/kaggle/working/dentalcare_models/dentex_fdi_v1.onnx",
    "/kaggle/working/dentalcare_models/dentex_disease_v1.onnx"
]:
    sess = ort.InferenceSession(model)

    print("\nMODELLO:", model)

    print("INPUT")
    for i in sess.get_inputs():
        print(i.name, i.shape, i.type)

    print("\nOUTPUT")
    for o in sess.get_outputs():
        print(o.name, o.shape, o.type)


MODELLO: /kaggle/working/dentalcare_models/dentex_fdi_v1.onnx
INPUT
images [1, 3, 1024, 1024] tensor(float)

OUTPUT
output0 [1, 36, 21504] tensor(float)

MODELLO: /kaggle/working/dentalcare_models/dentex_disease_v1.onnx
INPUT
images [1, 3, 1024, 1024] tensor(float)

OUTPUT
output0 [1, 8, 21504] tensor(float)


In [35]:
import json, shutil, os
from pathlib import Path

EXPORT_DIR = Path("/kaggle/working/dentalcare_final_export")
EXPORT_DIR.mkdir(exist_ok=True)

FDI_PT = "/kaggle/working/runs/dentex_fdi_v1-2/weights/best.pt"
DISEASE_PT = "/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt"

files = {
    "dentex_fdi_v1.onnx": "/kaggle/working/dentalcare_models/dentex_fdi_v1.onnx",
    "dentex_disease_v1.onnx": "/kaggle/working/dentalcare_models/dentex_disease_v1.onnx",
    "dentex_fdi_v1.pt": FDI_PT,
    "dentex_disease_v1.pt": DISEASE_PT,
    "fdi.yaml": "/kaggle/working/fdi.yaml",
    "disease.yaml": "/kaggle/working/disease.yaml",
}

for dst_name, src in files.items():
    shutil.copy2(src, EXPORT_DIR / dst_name)

metadata = {
    "project": "DentalCare",
    "models": {
        "fdi": {
            "name": "dentex_fdi_v1",
            "input": [1, 3, 1024, 1024],
            "output": [1, 36, 21504],
            "classes": 32
        },
        "disease": {
            "name": "dentex_disease_v1",
            "input": [1, 3, 1024, 1024],
            "output": [1, 8, 21504],
            "classes": {
                "0": "Impacted",
                "1": "Caries",
                "2": "Periapical_Lesion",
                "3": "Deep_Caries"
            }
        }
    }
}

with open(EXPORT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Export folder:")
for p in EXPORT_DIR.iterdir():
    print(p.name, round(p.stat().st_size / 1024 / 1024, 2), "MB")

Export folder:
dentex_fdi_v1.onnx 166.96 MB
dentex_disease_v1.onnx 42.93 MB
disease.yaml 0.0 MB
fdi.yaml 0.0 MB
metadata.json 0.0 MB
dentex_disease_v1.pt 21.51 MB
dentex_fdi_v1.pt 83.66 MB


In [36]:
!zip -r /kaggle/working/dentalcare_final_export.zip /kaggle/working/dentalcare_final_export

  adding: kaggle/working/dentalcare_final_export/ (stored 0%)
  adding: kaggle/working/dentalcare_final_export/dentex_fdi_v1.onnx (deflated 18%)
  adding: kaggle/working/dentalcare_final_export/dentex_disease_v1.onnx (deflated 16%)
  adding: kaggle/working/dentalcare_final_export/disease.yaml (deflated 22%)
  adding: kaggle/working/dentalcare_final_export/fdi.yaml (deflated 52%)
  adding: kaggle/working/dentalcare_final_export/metadata.json (deflated 65%)
  adding: kaggle/working/dentalcare_final_export/dentex_disease_v1.pt (deflated 8%)
  adding: kaggle/working/dentalcare_final_export/dentex_fdi_v1.pt (deflated 8%)


In [37]:
from pathlib import Path
import zipfile

zip_path = Path("/kaggle/working/dentalcare_final_export.zip")

print("ZIP esiste:", zip_path.exists())
print("Dimensione ZIP:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

with zipfile.ZipFile(zip_path, "r") as z:
    bad = z.testzip()
    print("Integrità ZIP:", "OK" if bad is None else f"ERRORE su {bad}")
    print("\nContenuto:")
    for name in z.namelist():
        print(name)

ZIP esiste: True
Dimensione ZIP: 270.23 MB
Integrità ZIP: OK

Contenuto:
kaggle/working/dentalcare_final_export/
kaggle/working/dentalcare_final_export/dentex_fdi_v1.onnx
kaggle/working/dentalcare_final_export/dentex_disease_v1.onnx
kaggle/working/dentalcare_final_export/disease.yaml
kaggle/working/dentalcare_final_export/fdi.yaml
kaggle/working/dentalcare_final_export/metadata.json
kaggle/working/dentalcare_final_export/dentex_disease_v1.pt
kaggle/working/dentalcare_final_export/dentex_fdi_v1.pt


In [38]:
from ultralytics import YOLO
from pathlib import Path

img = str(sorted(Path("/kaggle/working/dentex_disease_yolo/images/val").glob("*"))[0])

print(img)

pt_model = YOLO("/kaggle/working/runs/dentex_disease_v1-3/weights/best.pt")
onnx_model = YOLO("/kaggle/working/dentalcare_models/dentex_disease_v1.onnx")

pt_result = pt_model.predict(
    source=img,
    imgsz=1024,
    conf=0.25,
    verbose=False
)

onnx_result = onnx_model.predict(
    source=img,
    imgsz=1024,
    conf=0.25,
    verbose=False
)

print("PT detections :", len(pt_result[0].boxes))
print("ONNX detections:", len(onnx_result[0].boxes))

/kaggle/working/dentex_disease_yolo/images/val/train_127.png
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Loading /kaggle/working/dentalcare_models/dentex_disease_v1.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 311ms
Prepared 1 package in 1.93s
Installed 1 package in 121ms
 + onnxruntime-gpu==1.27.0

requirements: AutoUpdate success ✅ 2.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.27.0 with CPUExecutionProvider
PT detections : 2
ONNX detections: 2


In [39]:
import pandas as pd

def boxes_to_df(result):
    boxes = result[0].boxes
    rows = []

    for i in range(len(boxes)):
        xyxy = boxes.xyxy[i].cpu().numpy().tolist()
        cls = int(boxes.cls[i].cpu().numpy())
        conf = float(boxes.conf[i].cpu().numpy())

        rows.append({
            "class_id": cls,
            "confidence": round(conf, 4),
            "x1": round(xyxy[0], 1),
            "y1": round(xyxy[1], 1),
            "x2": round(xyxy[2], 1),
            "y2": round(xyxy[3], 1),
        })

    return pd.DataFrame(rows)

pt_df = boxes_to_df(pt_result)
onnx_df = boxes_to_df(onnx_result)

print("PT")
display(pt_df)

print("ONNX")
display(onnx_df)

PT


,class_id,confidence,x1,y1,x2,y2
0,1,0.8367,656.3,687.4,899.1,951.7
1,1,0.3163,1820.0,449.9,1961.4,746.1


ONNX


,class_id,confidence,x1,y1,x2,y2
0,1,0.8307,656.0,687.1,899.3,953.6
1,1,0.3313,1819.9,451.5,1961.4,745.9
